# 3. Observe a failure and replan

Goals: detect a stale action, preserve the observed physical state and obtain a recovery plan.

These are symbolic planning exercises. Gazebo is a separate static scene; these cells do not command a robot. Run cells in order with Python 3.

In [ ]:
from pathlib import Path
import sys
# Works from the student folder, solutions folder, or notebook server root.
base = Path.cwd().resolve()
for candidate in (base, *base.parents, base / 'jupyter_notebooks'):
    if (candidate / 'robo_restaurant' / 'planning.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError('Open this notebook inside the repository')
from dataclasses import replace
from robo_restaurant.planning import State, Action, solve, execute, successors, goal


## Demonstration

In [ ]:
initial = State()
stale_plan, _ = solve(initial)
observed = initial
while stale_plan[0].name != 'prepare':
    observed = execute(observed, stale_plan.pop(0))
observed = replace(observed, cooker_ok=False)
print('Cooker failure observed at:', observed)
try:
    execute(observed, stale_plan[0])
except ValueError as error:
    print('Expected stale-action rejection:', error)
else:
    raise AssertionError('Preparation on a broken cooker must fail')

## Your task

Implement `recover(observed)` returning `status`, `plan`, and `cost`. Solve from the observed state, not from the original initial state. For a feasible plan return status `ready`; otherwise return `blocked` with plan/cost `None`. Replay a feasible plan to verify goal satisfaction and its cost before returning it.

In [ ]:
def recover(observed):
    # TODO: replace this body.
    raise NotImplementedError


## Check your answer

The starter runs without errors but reports an unfinished task. After implementing it, this cell must report PASS.

In [ ]:
try:
    result = recover(observed)
    assert result['status'] == 'ready' and result['cost'] == 28
    assert result['plan'][0] == Action('repair')
    state = observed
    for action in result['plan']:
        state = execute(state, action)
    assert goal(state)
    assert recover(State(location='table_1', battery=0, cooker_ok=False)) == {
        'status': 'blocked', 'plan': None, 'cost': None}
    print('Total executed cost: travel before failure 2 + recovery 28 = 30')
except NotImplementedError:
    print("TODO: implement the exercise; this is not a pass")
else:
    print("PASS")

## Discussion

Which preparation precondition changed? Why must recovery preserve battery and location? The event is external, not a planner action. What changes when failure can happen during an action?